In [2]:
!pip install timm opencv-python

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import numpy as np
from PIL import Image
import os
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = "/kaggle/input/YOUR-DATASET-NAME"  # CHANGE
BATCH_SIZE = 8
EPOCHS = 60
LR = 3e-4
NUM_CLASSES = 10
IMG_H, IMG_W = 448, 896

print("Using:", DEVICE)

Using: cuda


In [4]:
value_map = {
    0: 0, 100: 1, 200: 2, 300: 3,
    500: 4, 550: 5, 700: 6,
    800: 7, 7100: 8, 10000: 9
}

def convert_mask(mask):
    arr = np.array(mask)
    new = np.zeros_like(arr, dtype=np.uint8)
    for k,v in value_map.items():
        new[arr == k] = v
    return Image.fromarray(new)

In [5]:
class SegDataset(Dataset):
    def __init__(self, root, train=True):
        self.img_dir = os.path.join(root, "Color_Images")
        self.mask_dir = os.path.join(root, "Segmentation")
        self.ids = os.listdir(self.img_dir)

        if train:
            self.img_tf = transforms.Compose([
                transforms.Resize((IMG_H, IMG_W)),
                transforms.RandomHorizontalFlip(),
                transforms.ColorJitter(0.3,0.3,0.3,0.1),
                transforms.ToTensor(),
                transforms.Normalize([0.485,0.456,0.406],
                                     [0.229,0.224,0.225])
            ])
        else:
            self.img_tf = transforms.Compose([
                transforms.Resize((IMG_H, IMG_W)),
                transforms.ToTensor(),
                transforms.Normalize([0.485,0.456,0.406],
                                     [0.229,0.224,0.225])
            ])

        self.mask_tf = transforms.Compose([
            transforms.Resize((IMG_H, IMG_W)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        name = self.ids[i]

        img = Image.open(os.path.join(self.img_dir, name)).convert("RGB")
        mask = convert_mask(Image.open(os.path.join(self.mask_dir, name)))

        img = self.img_tf(img)
        mask = (self.mask_tf(mask)*255).long().squeeze(0)

        return img, mask

In [5]:
ls

In [6]:
cd ..

/kaggle


In [7]:
ls

input/  lib/  working/


In [8]:
cd input

/kaggle/input


In [9]:
ls

datasets/


In [10]:
ls

datasets/


In [11]:
cd datasets

/kaggle/input/datasets


In [12]:
ls

roheeeeet/


In [13]:
cd roheeeeet

/kaggle/input/datasets/roheeeeet


In [14]:
ls

rohit-xeno/


In [15]:
cd rohit-xeno

/kaggle/input/datasets/roheeeeet/rohit-xeno


In [5]:
import os

print(os.listdir("/kaggle/input"))

['datasets']


In [7]:
import os
print(os.listdir("/kaggle/input/datasets"))

['roheeeeet']


In [22]:
print(os.listdir("/kaggle/input/datasets/roheeeeet"))

['rohit-xeno']


In [23]:
DATA_DIR = "/kaggle/input/datasets/roheeeeet/rohit-xeno"

In [24]:
print(os.listdir(DATA_DIR))

['Offroad_Segmentation_testImages', 'Offroad_Segmentation_Training_Dataset']


In [27]:
print(os.listdir("/kaggle/input/datasets/roheeeeet/rohit-xeno/Offroad_Segmentation_Training_Dataset"))


['Offroad_Segmentation_Training_Dataset']


In [38]:
print(os.listdir("/kaggle/input/datasets/roheeeeet/rohit-xeno/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset"))

['val', 'train']


In [29]:
ls

Offroad_Segmentation_testImages/  Offroad_Segmentation_Training_Dataset/


In [30]:
cd Offroad_Segmentation_Training_Dataset/

/kaggle/input/datasets/roheeeeet/rohit-xeno/Offroad_Segmentation_Training_Dataset


In [31]:
ls

Offroad_Segmentation_Training_Dataset/


In [32]:
cd Offroad_Segmentation_Training_Dataset


/kaggle/input/datasets/roheeeeet/rohit-xeno/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset


In [33]:
ls

train/  val/


In [7]:
train_ds = SegDataset(DATA_DIR + "/train", train=True)
val_ds   = SegDataset(DATA_DIR + "/val", train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(len(train_ds), len(val_ds))

2857 317


In [6]:
DATA_DIR = "/kaggle/input/datasets/roheeeeet/rohit-xeno/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset"

In [40]:
pwd

'/kaggle/input/datasets/roheeeeet/rohit-xeno/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset'

In [41]:
ls

train/  val/


In [8]:
backbone = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14")
backbone.to(DEVICE)
backbone.eval()

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 215MB/s]


DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
    (norm): Identity()
  )
  (blocks): ModuleList(
    (0-11): 12 x NestedTensorBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): MemEffAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (drop_path1): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): LayerScale()
      (drop_path2): Identity()
    )
  )
  (norm): LayerNorm((384,), eps=1e-06, elementwise_affi

In [9]:
class SegFormerHead(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, 256, 1)
        self.bn = nn.BatchNorm2d(256)

        self.conv2 = nn.Conv2d(256, 128, 3, padding=1)
        self.conv3 = nn.Conv2d(128, num_classes, 1)

    def forward(self, x, H, W):
        B, N, C = x.shape
        x = x.reshape(B, H, W, C).permute(0,3,1,2)

        x = F.relu(self.bn(self.conv1(x)))
        x = F.relu(self.conv2(x))
        return self.conv3(x)

In [10]:
dummy = torch.randn(1,3,IMG_H,IMG_W).to(DEVICE)
feat = backbone.forward_features(dummy)["x_norm_patchtokens"]
EMB = feat.shape[-1]

model = SegFormerHead(EMB, NUM_CLASSES).to(DEVICE)

In [11]:
class DiceLoss(nn.Module):
    def forward(self, pred, target):
        pred = F.softmax(pred, dim=1)
        target = F.one_hot(target, NUM_CLASSES).permute(0,3,1,2).float()

        inter = (pred * target).sum((2,3))
        union = pred.sum((2,3)) + target.sum((2,3))
        dice = (2*inter + 1e-6)/(union + 1e-6)
        return 1 - dice.mean()

dice_loss = DiceLoss()

def loss_fn(pred, target):
    ce = F.cross_entropy(pred, target)
    return 0.5*ce + 0.5*dice_loss(pred, target)

In [12]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

In [13]:
def compute_iou(pred, target):
    pred = torch.argmax(pred,1)
    return (pred==target).float().mean().item()

In [15]:
EPOCHS = 10
best_iou = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for img, mask in tqdm(train_loader):
        img, mask = img.to(DEVICE), mask.to(DEVICE)

        with torch.no_grad():
            feat = backbone.forward_features(img)["x_norm_patchtokens"]

        out = model(feat, IMG_H//14, IMG_W//14)
        out = F.interpolate(out, size=(IMG_H, IMG_W))

        loss = loss_fn(out, mask)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # ===== VALIDATION WITH TTA =====
    model.eval()
    ious = []

    with torch.no_grad():
        for img, mask in val_loader:
            img, mask = img.to(DEVICE), mask.to(DEVICE)

            # ORIGINAL
            feat = backbone.forward_features(img)["x_norm_patchtokens"]
            out1 = model(feat, IMG_H//14, IMG_W//14)
            out1 = F.interpolate(out1, size=(IMG_H, IMG_W))

            # FLIP
            img_flip = torch.flip(img, dims=[3])
            feat_flip = backbone.forward_features(img_flip)["x_norm_patchtokens"]

            out2 = model(feat_flip, IMG_H//14, IMG_W//14)
            out2 = F.interpolate(out2, size=(IMG_H, IMG_W))
            out2 = torch.flip(out2, dims=[3])

            # FINAL
            out = (out1 + out2) / 2

            ious.append(compute_iou(out, mask))

    val_iou = np.mean(ious)

    print(f"Epoch {epoch+1}/15 | Loss {total_loss:.3f} | IoU {val_iou:.4f}")

    # SAVE BEST MODEL
    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), "/kaggle/working/best_model.pth")
        print("Best model saved!")

print("FINAL BEST IOU:", best_iou)

100%|██████████| 358/358 [03:24<00:00,  1.75it/s]


Epoch 1/15 | Loss 337.707 | IoU 0.7251
Best model saved!


100%|██████████| 358/358 [03:23<00:00,  1.76it/s]


Epoch 2/15 | Loss 316.408 | IoU 0.7309
Best model saved!


100%|██████████| 358/358 [03:23<00:00,  1.76it/s]


...

Epoch 8/15 | Loss 295.125 | IoU 0.7328


100%|██████████| 358/358 [03:23<00:00,  1.76it/s]


Epoch 9/15 | Loss 293.128 | IoU 0.7324


100%|██████████| 358/358 [03:23<00:00,  1.76it/s]


Epoch 10/15 | Loss 292.301 | IoU 0.7296
FINAL BEST IOU: 0.7371811851859092


In [20]:
pwd

'/kaggle/working'

In [21]:
ls

best_model.pth


In [16]:
# =========================
# LOAD MODEL
# =========================
model.load_state_dict(torch.load("/kaggle/working/best_model.pth"))
model.eval()
print("Model loaded")

Model loaded


In [17]:
def compute_real_iou(pred, target, num_classes=10):
    pred = torch.argmax(pred, dim=1)

    ious = []
    for cls in range(num_classes):
        pred_inds = (pred == cls)
        target_inds = (target == cls)

        intersection = (pred_inds & target_inds).sum().float()
        union = (pred_inds | target_inds).sum().float()

        if union == 0:
            continue

        ious.append((intersection / union).item())

    return np.mean(ious) if len(ious) > 0 else 0

In [18]:
# =========================
# TESTING LOOP
# =========================
test_ious = []

with torch.no_grad():
    for img, mask in tqdm(val_loader):   # val_loader = test_loader if separate
        img, mask = img.to(DEVICE), mask.to(DEVICE)

        # ===== ORIGINAL =====
        feat = backbone.forward_features(img)["x_norm_patchtokens"]
        out1 = model(feat, IMG_H//14, IMG_W//14)
        out1 = F.interpolate(out1, size=(IMG_H, IMG_W))

        # ===== FLIP TTA =====
        img_flip = torch.flip(img, dims=[3])
        feat_flip = backbone.forward_features(img_flip)["x_norm_patchtokens"]

        out2 = model(feat_flip, IMG_H//14, IMG_W//14)
        out2 = F.interpolate(out2, size=(IMG_H, IMG_W))
        out2 = torch.flip(out2, dims=[3])

        # ===== FINAL =====
        out = (out1 + out2) / 2

        iou = compute_real_iou(out, mask)
        test_ious.append(iou)

# =========================
# FINAL RESULT
# =========================
mean_iou = np.mean(test_ious)

print("\n TEST RESULTS")
print("="*40)
print(f"Mean IoU: {mean_iou:.4f}")
print("="*40)

100%|██████████| 40/40 [00:41<00:00,  1.05s/it]


 TEST RESULTS
Mean IoU: 0.3598


In [19]:
import torch
import torch.nn.functional as F
import cv2
import numpy as np
import os
from tqdm import tqdm

# =========================
# CONFIG
# =========================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR = "/kaggle/working/visualizations"
os.makedirs(SAVE_DIR, exist_ok=True)

# =========================
# FIXED COLOR MAP (IMPORTANT)
# =========================
COLOR_MAP = {
    0: [0, 0, 0],        # background
    1: [0, 255, 0],      # trees
    2: [34,139,34],      # bushes
    3: [189,183,107],    # dry grass
    4: [139,69,19],      # dry bushes
    5: [210,180,140],    # clutter
    6: [160,82,45],      # logs
    7: [128,128,128],    # rocks
    8: [0,100,0],        # landscape
    9: [135,206,235]     # sky
}

# =========================
# COLORIZE MASK
# =========================
def colorize(mask):
    h, w = mask.shape
    color_mask = np.zeros((h, w, 3), dtype=np.uint8)

    for cls, color in COLOR_MAP.items():
        color_mask[mask == cls] = color

    return color_mask

# =========================
# MAIN VISUALIZATION LOOP
# =========================
model.eval()

with torch.no_grad():
    for idx, (img, mask) in enumerate(tqdm(val_loader)):

        img = img.to(DEVICE)

        # ===== PREDICTION =====
        feat = backbone.forward_features(img)["x_norm_patchtokens"]
        out = model(feat, IMG_H//14, IMG_W//14)
        out = F.interpolate(out, size=(IMG_H, IMG_W))

        pred = torch.argmax(out, dim=1).cpu().numpy()[0]
        gt   = mask.numpy()[0]

        # ===== ORIGINAL IMAGE =====
        img_np = img.cpu().numpy()[0]
        img_np = np.transpose(img_np, (1,2,0))

        # denormalize
        mean = np.array([0.485,0.456,0.406])
        std  = np.array([0.229,0.224,0.225])
        img_np = (img_np * std + mean) * 255
        img_np = img_np.astype(np.uint8)

        # ===== COLOR MASKS =====
        pred_color = colorize(pred)
        gt_color   = colorize(gt)

        # ===== OVERLAY =====
        overlay = cv2.addWeighted(img_np, 0.6, pred_color, 0.4, 0)

        # ===== CONCAT VIEW =====
        combined = np.concatenate([
            img_np,
            gt_color,
            pred_color,
            overlay
        ], axis=1)

        # ===== SAVE =====
        save_path = os.path.join(SAVE_DIR, f"{idx}.png")
        cv2.imwrite(save_path, combined)

print("Visualization saved at:", SAVE_DIR)

100%|██████████| 40/40 [00:27<00:00,  1.46it/s]

Visualization saved at: /kaggle/working/visualizations
